## Read Source Data

In [1]:
from datetime import datetime, timezone
from uuid import uuid8

import numpy as np
import pandas as pd
from geojson import Polygon
from trap_schema.datapackage import Contributor, DataPackage, License, Project, RelatedIdentifiers, Source, Taxonomic, Temporal
from trap_schema.tables import DeploymentRow, DeploymentTable, MediaRow, MediaTable, ObservationsRow, ObservationsTable
from trap_schema.transform import (
    ColumnMap,
    Transformer,
    calc_height,
    calc_width,
    hardcode,
    parse_wkt_point_latitude,
    parse_wkt_point_longitude,
    timestamp_to_iso,
)

# Source data
df = pd.read_parquet("../raw-data/metadata.parquet")
df = df[df.sourceimageid.notna()].replace({np.nan: None, pd.NA: None}) # Source image is required in CamtrapDP


### `observations.csv`

In [2]:
observations_columns = [
    "detectionid", "sourceimageid", "deploymentid",
    "timestamp",
    "x1", "x2", "y1", "y2",
    "label", "score", "algorithm",
    "taxonlevel"
]
observations_transform = Transformer([
    ColumnMap("x2", "bboxWidth", calc_width(4096)),
    ColumnMap("y2", "bboxHeight", calc_height(2160)),
    ColumnMap("detectionid", "observationID"),
    ColumnMap("sourceimageid", "mediaID"),
    ColumnMap("deploymentid", "deploymentID"),
    ColumnMap("label", "scientificName"),
    ColumnMap("score", "classificationProbability", lambda x, y : x / 100),
    ColumnMap("algorithm", "classifiedBy"),
    ColumnMap("x1", "bboxX", lambda x, y : x / 4096),
    ColumnMap("y1", "bboxY", lambda x, y : x / 2160),
    ColumnMap("timestamp", "eventStart", timestamp_to_iso),
    ColumnMap("timestamp", "eventEnd", timestamp_to_iso),
    ColumnMap(None, "observationLevel", hardcode("media")),
    ColumnMap(None, "observationType", hardcode("animal")),
])

rank_order = ("species", "genus", "family", "order", "species")

def summarize_detection(df : pd.DataFrame, threshold : float | list[float] | dict[str, float]):
    df = df.set_index("taxonlevel").sort_index(key=lambda x : x.map(rank_order.index))
    df = assign_threshold(df, threshold)
    above_thr = df["score"] > df["threshold"]
    n_above = above_thr.sum()
    if n_above > 0:
        df = df.loc[above_thr]
    if len(df) > 1:
        df = df.iloc[0]
    assert isinstance(df, pd.Series), f'{df}'
    out = observations_transform(df)
    if n_above == 0:
        out.pop("scientificName")
        out.pop("classificationProbability")
        out.pop("classifiedBy")
    return out

def assign_threshold(df : pd.DataFrame, threshold : float | list[float] | dict[str, float]):
    if isinstance(threshold, float):
        threshold = {alg : threshold for alg in df["algorithm"].unique()}
    elif isinstance(threshold, list):
        threshold = {alg : thr for alg, thr in zip(df["algorithm"], threshold)}
    df["threshold"] = pd.Series([threshold[alg] for alg in df["algorithm"]])
    return df

obs_rows = []
for did, grp in df[observations_columns].groupby(["detectionid"]):
    grp_sum = summarize_detection(grp, 0.1)
    obs_rows.append(ObservationsRow.from_dict(grp_sum))

observations_table = ObservationsTable(rows=obs_rows)

### `media.csv`

In [3]:
media_columns = [
    "sourceimageid", "deploymentid",
    "filename", "url",
    "timestamp"
]

media_transform = Transformer([
    ColumnMap("sourceimageid", "mediaID"),
    ColumnMap("deploymentid", "deploymentID"),
    ColumnMap("url", "filePath"), # confusingly this usually refers to a URL, not a literal path
    ColumnMap("filename", "fileName"),
    ColumnMap("url", "fileMediatype", lambda x, y : "image/" + x.split(".")[-1]),
    ColumnMap("timestamp", "timestamp", timestamp_to_iso),
    ColumnMap(None, "filePublic", hardcode(True)),
])

def summarize_media(df : pd.DataFrame):
    df.drop_duplicates(inplace=True)
    return media_transform(df)

media_rows = []
for siid, grp in df[media_columns].groupby(["sourceimageid"]):
    grp_sum = summarize_media(grp)
    media_rows.append(MediaRow.from_dict(grp_sum))

media_table = MediaTable(rows=media_rows)

### `deployments.csv`

In [4]:
deployment_columns = [
    "code", "deploymentid",
    "wktposition", "timestamp"
]

deployment_transform = Transformer([
    ColumnMap("code", "locationName"),
    ColumnMap("deploymentid", "deploymentID"),
    ColumnMap("wktposition", "longitude", parse_wkt_point_longitude),
    ColumnMap("wktposition", "latitude", parse_wkt_point_latitude),
    ColumnMap("timestamp_min", "deploymentStart", timestamp_to_iso),
    ColumnMap("timestamp_max", "deploymentEnd", timestamp_to_iso),
])

def summarize_deployment(df : pd.DataFrame):
    ts = df["timestamp"].dropna() # dropna is not necessary, but included for robustness
    tsmin = ts.min()
    tsmax = ts.max()
    df.drop("timestamp", axis="columns", inplace=True)
    df.drop_duplicates(inplace=True)
    df["timestamp_min"] = [tsmin]
    df["timestamp_max"] = [tsmax]
    return deployment_transform(df)

deployment_rows = []
for siid, grp in df[deployment_columns].groupby(["deploymentid"]):
    grp_sum = summarize_deployment(grp)
    deployment_rows.append(DeploymentRow.from_dict(grp_sum))

deployment_table = DeploymentTable(rows=deployment_rows)

### `datapackage.json`

In [5]:
coords = df['wktposition'].str.extract(r'POINT\(([-\d.]+) ([-\d.]+)\)').astype(float)
coords.columns = ['lon', 'lat']

# 2. Find the minimum and maximum boundaries
min_lon, max_lon = coords['lon'].min().item(), coords['lon'].max().item()
min_lat, max_lat = coords['lat'].min().item(), coords['lat'].max().item()

# 3. Define the bounding box corners 
# A valid GeoJSON polygon requires a closed loop (start and end at the same point)
bbox_coords = [
    (min_lon, min_lat), # Bottom-left
    (min_lon, max_lat), # Top-left
    (max_lon, max_lat), # Top-right
    (max_lon, min_lat), # Bottom-right
    (min_lon, min_lat)  # Back to Bottom-left
]

# 4. Create the GeoJSON Polygon and dump it to a string
bbox_polygon = Polygon([bbox_coords])

In [6]:
contributors = [
    Contributor(
        title="Toke Thomas Høje",
        email="tth@ecos.au.dk",
        path="https://pure.au.dk/portal/en/persons/tth%40ecos.au.dk/",
        role="principalInvestigator",
        organization="Aarhus University, Department of Ecoscience"
    ),
    Contributor(
        title="Lars Dalby",
        email="lars@ecos.au.dk",
        path="https://pure.au.dk/portal/da/persons/lars%40ecos.au.dk/",
        role="contributor",
        organization="Aarhus University, Department of Ecoscience"
    ),
    Contributor(
        title="Asger Svenning",
        email="asgersvenning@ecos.au.dk",
        path="https://asgersvenning.com",
        role="contributor",
        organization="Aarhus University, Department of Ecoscience"
    )
]

sources = [
    Source(
        title="IAS database"
    )
]

taxonomic = [
    Taxonomic(
        scientificName=record["label"],
        taxonRank=record["taxonlevel"],
        taxonID=record["labelid"]
    ) for record in df[["label", "taxonlevel", "labelid"]].drop_duplicates().to_dict(orient="records")
]
licenses = [
    License(
      name="CC0-1.0",
      scope="data"
    ),
    License(
      path="http://creativecommons.org/licenses/by/4.0/",
      scope="media"
    )
]

datapackage = DataPackage(
    name="demo-ias-camtrap-dp-1.0.2",
    id=str(uuid8()),
    created=datetime.now(timezone.utc),
    title="Demo IAS (Invasive Alien Species) dataset in CamTrapDP 1.0.2",
    contributors=contributors,
    description=""""Demo of a standardized IAS (Invasive Alien Species) dataset.
    Constructed based on a relatively arbitrary dump from the IAS database.
    Created for Workgroup 3 datathon (workshop) in the InsectAI COST Action at the annual meeting in 2026, Niš, Serbia.
    The demo and the associated worked example has been created mainly by Asger Svenning in collaboration with Lars Dalby and others.""",
    version="0.1.0",
    keywords=["ias", "insectai", "camtrapdp", "demo", "timelapse", "ami"],
    sources=sources,
    licenses=licenses,
    project=Project(
        title="Invasive Alien Species",
        acronym="IAS",
        samplingDesign="systematicRandom",
        captureMethod=["timeLapse"],
        individualAnimals=False,
        observationLevel=["media"]
    ),
    spatial=bbox_polygon,
    temporal=Temporal(
        start=df["timestamp"].min().item(),
        end=df["timestamp"].max().item()
    ),
    taxonomic=taxonomic
)


## Export

In [7]:
output_dir = ".."
datapackage.save(output_dir)
observations_table.save(output_dir)
media_table.save(output_dir)
deployment_table.save(output_dir)

'../deployments.csv'

In [8]:
!uvx frictionless validate ../datapackage.json

⠙ idna==3.13                                                                    /home/au644314/.cache/uv/archive-v0/RPTS0iPkVejOHiS5aEuLO/lib/python3.14/site-packages/frictionless/metadata/metadata.py:549: DeprecationWarning: Automatically retrieving remote references can be a security vulnerability and is discouraged by the JSON Schema specifications. Relying on this behavior is deprecated and will shortly become an error. If you are sure you want to remotely retrieve your reference and that it is safe to do so, you can find instructions for doing so via referencing.Registry in the referencing documentation (https://referencing.readthedocs.org).
  for error in validator.iter_errors(descriptor):  # type: ignore
─────────────────────────────────── Dataset ────────────────────────────────────
                      dataset                       
┏━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ name         ┃ type  ┃ path             ┃ status ┃
┡━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━━━━━━━